In [1]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import Prompts as prompts


In [ ]:
import importlib
importlib.reload(prompts)

In [2]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [3]:
from Prompts.prompts import Config

In [4]:
# Load your CSV
df_new = pd.read_csv("/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Amazon_Human_VS_ComputerFake.csv")
reviews = df_new["text"].tolist()

In [44]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [45]:
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


In [ ]:
apptainer --version

In [ ]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11400"

In [ ]:
# Classify each review
results = []
for r in reviews:
    result = chain.invoke({"text": r})
    # Clean the output - extract only "truthful" or "deceptive"
    label = result.strip().lower()
    # Extract first word if model adds extra text
    if " " in label:
        label = label.split()[0]
    # Remove any punctuation
    label = label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {label}")
    results.append(label)

In [46]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real. -> Cleaned: real
Raw output: Real. 

The language used seems genuine, but there are inconsistencies that raise suspicions. The review contains repetitive phrases ("I would recommend", "she is a very tall woman", "it looks good in her bed") and similar sentences regarding the purchase for multiple friends. It also includes an outlandish statement about the product being difficult to assemble and not fitting the reviewer, which seems unlikely given the context of buying it as a gift for different individuals. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw outp

In [47]:
df_new.head(10)


,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1
5,fake,Home_and_Kitchen,amazon,Bought this on a whim and it was the best purc...,1
6,fake,Home_and_Kitchen,amazon,"I admit I was leary of the suction, but I did ...",1
7,fake,Home_and_Kitchen,amazon,Purchased one as a gift and it was a very nice...,1
8,fake,Home_and_Kitchen,amazon,I wanted something to hold a bunch of pieces o...,1
9,fake,Home_and_Kitchen,amazon,I'm so happy I found this set and am very plea...,1


In [48]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [ ]:
y_true = df_new["Binary_label"].tolist()

In [ ]:
for label in results:
    print(label)

In [49]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

ZERO-SHOT PROMPTING RESULTS
Accuracy: 0.5675

F1 Score: 0.4933780410129511

Confusion Matrix:
[[ 74 326]
 [ 20 380]]


## One-Shot Prompting
Using one example to guide the model's classification

In [6]:
llm = OllamaLLM(model=Config.model)

In [7]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [8]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfect for my kids. I love them.These are super lightweight and lightweight. I have used them for a couple of months and they are still working great. I would definitely recomme

In [ ]:
# # Classify each review
# one_shot_results = []
# for r in reviews:
#     result = one_shot_chain.invoke({"text": r})
#     # Clean the output - extract only "truthful" or "deceptive"
#     label = result.strip().lower()
#     # Extract first word if model adds extra text
#     if " " in label:
#         label = label.split()[0]
#     # Remove any punctuation
#     label = label.strip('.,!?;:')
#     print(f"Raw output: {result} -> Cleaned: {label}")
#     one_shot_results.append(label)

In [10]:
# Classify each review
one_shot_results = []

for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


Raw output: Real -> Cleaned: real
Raw output: Real. -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw outpu

In [ ]:
df_new.head(10)

In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [12]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS
Accuracy: 0.6175

F1 Score: 0.616

Confusion Matrix:
[[272 128]
 [178 222]]


## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [5]:
llm = OllamaLLM(model=Config.model)

In [6]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [7]:
print(Config.few_shot_prompt_template)


    You are an expert at detecting real and fake reviews.


    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Example 1:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfect for my kids. I love them.These are super lightweight and lightweight. I have used them for a couple of months and they are still working gr

In [ ]:
# Classify reviews using few-shot prompting
# few_shot_results = []
# for r in reviews:
#     result = few_shot_chain.invoke({"text": r})
#     # Clean the output
#     label = result.strip().lower()
#     if " " in label:
#         label = label.split()[0]
#     label = label.strip('.,!?;:')
#     print(f"Raw output: {result} -> Cleaned: {label}")
#     few_shot_results.append(label)

In [9]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: 

In [12]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS
Accuracy: 0.63375

F1 Score: 0.6040801366126218

Confusion Matrix:
[[363  37]
 [256 144]]


## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [ ]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")